In [ ]:
# =============================================================================
#  片段 1: 环境设置与库导入
# =============================================================================
# 确保已安装最新版本的库
!pip install earthengine-api pandas requests --upgrade

import ee
import pandas as pd
import requests
from datetime import datetime
import time

print("✅ 片段 1: 库已成功安装并导入。")

✅ 片段 1: 库已成功安装并导入。


In [ ]:
# =============================================================================
#  片段 2: GEE授权与初始化
# =============================================================================
try:
    # 尝试使用已有的凭证和项目进行初始化
    # 请将 'YOUR_GEE_PROJECT' 替换为您自己的GEE项目ID
    ee.Initialize(project='awaran-20130924')
    print("✅ GEE 初始化成功！")
except Exception as e:
    # 如果初始化失败，则提示进行授权
    print("GEE 需要授权...")
    ee.Authenticate()
    ee.Initialize(project='awaran-20130924')
    print("✅ GEE 授权并初始化成功！")

GEE 需要授权...
✅ GEE 授权并初始化成功！


In [ ]:
# =============================================================================
#  片段 3: 获取并筛选地震数据 (【已修改】6.5级以上 & 2012年之后)
# =============================================================================
import pandas as pd
from datetime import datetime
import requests

def fetch_and_filter_earthquakes(start_year=2012, min_magnitude=6.5): # <-- 【核心修改点】
    """从USGS获取指定震级和起始年份以上的地震数据。"""

    start_time = f"{start_year}-01-01"

    print(f"正在从USGS获取 {start_time} 以来, {min_magnitude} 级以上的大地震数据...")
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    params = {
        'format': 'geojson',
        'starttime': start_time,
        'endtime': datetime.now().strftime('%Y-%m-%d'),
        'minmagnitude': min_magnitude,
        'orderby': 'time-asc'
    }
    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        earthquakes = []
        for feature in data['features']:
            properties = feature['properties']
            geometry = feature['geometry']
            eq_time = datetime.fromtimestamp(properties['time'] / 1000)
            earthquakes.append({
                'date': eq_time.strftime('%Y-%m-%d'),
                'magnitude': properties['mag'],
                'location_name': properties['place'],
                'longitude': geometry['coordinates'][0],
                'latitude': geometry['coordinates'][1]
            })
        print(f"✅ 成功获取到 {len(data['features'])} 次原始事件。")
        return earthquakes
    except requests.exceptions.RequestException as e:
        print(f"🔴 错误: 无法从USGS获取地震数据: {e}")
        return []

# 执行函数
raw_earthquakes = fetch_and_filter_earthquakes(start_year=2012, min_magnitude=6.5)

正在从USGS获取 2012-01-01 以来, 6.5 级以上的大地震数据...
✅ 成功获取到 605 次原始事件。


In [ ]:
# =============================================================================
#  片段 3.5: 云端预筛选 (【已修改】人口阈值 > 5万)
# =============================================================================
def pre_filter_earthquakes_for_urban_impact(earthquakes, pop_threshold=50000, buffer_radius=200000): # <-- 【核心修改点】
    """
    在GEE服务器端，根据影响区内的人口数量，预先筛选掉无效的地震。
    """
    if not earthquakes:
        return []

    print(f"\n--- 开始在GEE上进行城市影响预筛选 (人口阈值: {pop_threshold/1e3:.0f} 千) ---")

    features = [ee.Feature(ee.Geometry.Point(eq['longitude'], eq['latitude']), eq) for eq in earthquakes]
    eq_collection = ee.FeatureCollection(features)

    population_image = ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.date('2020-01-01', '2020-12-31')).mean()

    def calculate_population(feature):
        aoi = feature.geometry().buffer(buffer_radius)
        pop_sum_dict = population_image.reduceRegion(
            reducer=ee.Reducer.sum(), geometry=aoi, scale=1000, maxPixels=1e9
        )
        return feature.set('population_in_aoi', pop_sum_dict.get('population'))

    pop_collection = eq_collection.map(calculate_population)
    filtered_collection = pop_collection.filter(ee.Filter.And(
        ee.Filter.notNull(['population_in_aoi']),
        ee.Filter.gt('population_in_aoi', pop_threshold)
    ))

    print("  -> 正在从GEE获取筛选结果...")

    size = filtered_collection.size()
    list_of_features = ee.List(ee.Algorithms.If(
        size.gt(0), filtered_collection.toList(size), ee.List([])
    ))

    filtered_list_of_dicts = list_of_features.getInfo()
    final_earthquakes = [f['properties'] for f in filtered_list_of_dicts]

    if not final_earthquakes:
        print("✅ 预筛选完成！即使放宽条件，在您的数据集中仍未发现对城市/城镇有潜在影响的地震。")
    else:
        print(f"✅ 预筛选完成！从 {len(earthquakes)} 次事件中筛选出 {len(final_earthquakes)} 次对城市/城镇有潜在影响的地震。")

    return final_earthquakes

# 执行预筛选
if 'raw_earthquakes' in locals():
    earthquakes_to_process = pre_filter_earthquakes_for_urban_impact(raw_earthquakes, pop_threshold=50000)
    if earthquakes_to_process:
        print("\n有效地震数据预览：")
        display(pd.DataFrame(earthquakes_to_process).head())


--- 开始在GEE上进行城市影响预筛选 (人口阈值: 50 千) ---
  -> 正在从GEE获取筛选结果...
✅ 预筛选完成！从 605 次事件中筛选出 90 次对城市/城镇有潜在影响的地震。

有效地震数据预览：


,date,latitude,location_name,longitude,magnitude,population_in_aoi
0,2012-02-06,9.999,"2 km NNE of Jimalalud, Philippines",123.206,6.7,154453.522674
1,2012-04-17,-32.625,"22 km NW of Hacienda La Calera, Chile",-71.365,6.7,100036.097532
2,2012-09-30,1.929,"11 km WNW of San Agustín, Colombia",-76.362,7.3,88866.707062
3,2012-11-07,13.988,"33 km S of Champerico, Guatemala",-91.895,7.4,121612.720188
4,2012-11-11,23.005,"51 km NNE of Shwebo, Myanmar",95.885,6.8,91266.828301


In [ ]:
# =============================================================================
#  片段 4: 核心GEE分析函数 (按周合成)
# =============================================================================
from datetime import datetime

def process_one_earthquake_on_server(feature):
    SEARCH_RADIUS = 200000
    NTL_CAP = 1000
    NTL_BAND_NAME = 'DNB_BRDF_Corrected_NTL'
    def mask_vnp_quality(image):
        qf_band = image.select('QF_Cloud_Mask')
        clear_mask = qf_band.bitwiseAnd(1 << 0).eq(0).And(qf_band.bitwiseAnd(1 << 1).eq(0))
        return image.updateMask(clear_mask).select(NTL_BAND_NAME)
    search_area = feature.geometry().buffer(SEARCH_RADIUS)
    urban_polygons = ee.Image('JRC/GHSL/P2023A/GHS_SMOD/2020').select('smod_code') \
                       .gte(2).selfMask().reduceToVectors(geometry=search_area, scale=500, maxPixels=1e10)
    def perform_analysis():
        urban_aoi = urban_polygons.union(100)
        start_date = ee.Date('2012-01-01')
        end_date = ee.Date('2025-01-01')
        n_weeks = end_date.difference(start_date, 'week').ceil()
        weeks = ee.List.sequence(0, n_weeks.subtract(1))
        def get_weekly_sum(week_offset):
            current_start = start_date.advance(week_offset, 'week')
            current_end = current_start.advance(1, 'week')
            weekly_image = ee.ImageCollection('NASA/VIIRS/002/VNP46A2') \
                              .filterBounds(search_area) \
                              .filterDate(current_start, current_end) \
                              .map(mask_vnp_quality) \
                              .median()
            def compute_sum(img):
                sum_dict = ee.Image(img).lte(NTL_CAP).reduceRegion(
                    reducer=ee.Reducer.sum(), geometry=urban_aoi, scale=500, maxPixels=1e13
                )
                return sum_dict.get(NTL_BAND_NAME, -999)
            sum_value = ee.Algorithms.If(weekly_image.bandNames().size().gt(0), compute_sum(weekly_image), -999)
            prop_name = ee.String('NTL_').cat(current_start.format('YYYY_ww'))
            return prop_name, sum_value
        time_series_list = weeks.map(get_weekly_sum)
        return feature.set(ee.Dictionary(time_series_list.flatten()))
    return ee.Feature(ee.Algorithms.If(urban_polygons.size().gt(0), perform_analysis(), None), feature.toDictionary())
print("✅ 片段 4: GEE核心分析函数已准备就绪 (按周合成)。")

✅ 片段 4: GEE核心分析函数已准备就绪 (按周合成)。


In [ ]:
# =============================================================================
#  片段 5: 服务器端批处理工作流与导出
# =============================================================================
import time

def server_side_batch_export_workflow(earthquakes, batch_size=5):
    if not earthquakes:
        print("\n🔴 经过预筛选，没有发现对城市/城镇有显著影响的地震，任务终止。")
        return
    num_earthquakes = len(earthquakes)
    print(f"\n--- 将对 {num_earthquakes} 次有效地震进行精确夜光分析 ---")
    total_batches = (num_earthquakes + batch_size - 1) // batch_size
    for i in range(0, num_earthquakes, batch_size):
        batch_num = (i // batch_size) + 1
        current_batch_data = earthquakes[i:i + batch_size]
        print(f"\n--- 正在准备批次 {batch_num} / {total_batches} (包含 {len(current_batch_data)} 个事件) ---")
        features = [ee.Feature(ee.Geometry.Point(eq['longitude'], eq['latitude']), eq) for eq in current_batch_data]
        eq_collection = ee.FeatureCollection(features)
        print("  -> 正在将计算任务发送到GEE服务器...")
        processed_collection = eq_collection.map(process_one_earthquake_on_server, opt_dropNulls=True)

        # 【文件名更新】
        base_filename = 'urban_earthquake_NTL_weekly_M65_pop50k'
        output_filename = f"{base_filename}_batch_{batch_num}_of_{total_batches}"

        task = ee.batch.Export.table.toDrive(
            collection=processed_collection, description=output_filename,
            folder='GEE_Earthquake_Exports', fileNamePrefix=output_filename, fileFormat='CSV'
        )
        task.start()
        print(f"🎉 任务 '{output_filename}' 已成功提交！")
        if total_batches > 1: time.sleep(5)
    print(f"\n\n✅ 所有 {total_batches} 个批处理任务均已提交。")

if 'earthquakes_to_process' in locals() and earthquakes_to_process:
    server_side_batch_export_workflow(earthquakes_to_process, batch_size=5)
else:
    print("\n🔴 未找到可处理的地震。请检查片段3和3.5的筛选逻辑。")


--- 将对 90 次有效地震进行精确夜光分析 ---

--- 正在准备批次 1 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_weekly_M65_pop50k_batch_1_of_18' 已成功提交！

--- 正在准备批次 2 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_weekly_M65_pop50k_batch_2_of_18' 已成功提交！

--- 正在准备批次 3 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_weekly_M65_pop50k_batch_3_of_18' 已成功提交！

--- 正在准备批次 4 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_weekly_M65_pop50k_batch_4_of_18' 已成功提交！

--- 正在准备批次 5 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_weekly_M65_pop50k_batch_5_of_18' 已成功提交！

--- 正在准备批次 6 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_weekly_M65_pop50k_batch_6_of_18' 已成功提交！

--- 正在准备批次 7 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_weekly_M65_pop50k_batch_7_of_18' 已成功提交！

--- 正在准备批次 8 / 18 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_earthquake_NTL_wee